In [14]:
import requests
import pandas as pd

import altair as alt
import lxml

In [22]:
league_id = 'kfszz7vdmdl0krou'
season = '15'
latest_gw = 6

In [10]:
records = []

for period in range(1, 39):  # periods 1–38
    # url= f"https://www.fantrax.com/fantasy/league/{league_id}/standings;view=REGULAR_SEASON;timeframeType=BY_PERIOD;timeStartType=FROM_SEASON_START;period={period}"
    url = f'https://www.fantrax.com/fxpa/req?leagueId={league_id}'

    headers = {
        "accept": "application/json",
        "content-type": "text/plain",
        "sec-ch-ua": "\"Not;A=Brand\";v=\"99\", \"Google Chrome\";v=\"139\", \"Chromium\";v=\"139\"",
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": "\"macOS\"",
        "referer": "https://www.fantrax.com/fantasy/league/962c1utjlzbm8vb8/standings;view=REGULAR_SEASON;timeframeType=BY_PERIOD;timeStartType=FROM_SEASON_START;period=38"
    }
    
    payload = {
        "msgs": [
            {
                "method": "getStandings",
                "data": {
                    "leagueId": league_id,
                    "view": "REGULAR_SEASON",
                    "timeframeType": "BY_PERIOD",
                    "timeStartType": "FROM_SEASON_START",
                    "period": str(period)
                }
            }
        ],
        "uiv": 3,
        "refUrl": f"https://www.fantrax.com/fantasy/league/{league_id}/standings;view=REGULAR_SEASON;timeframeType=BY_PERIOD;timeStartType=FROM_SEASON_START;period={period}",
        "dt": 1,
        "at": 0,
        "av": "0.0",
        "tz": "America/Los_Angeles",
        "v": "167.0.1"
    }

    r = requests.post(url, headers=headers, json=payload)
    r.raise_for_status()
    j = r.json()
    

    data = j["responses"][0]["data"]
    team_info = data.get("fantasyTeamInfo", {})

    # find the "Standings" table
    standings_tbl = next(
        (t for t in data.get("tableList", []) if t.get("caption") == "Standings"),
        None
    )
    if not standings_tbl:
        print(f"Period {period}: Standings table not found")
        continue

    for row in standings_tbl.get("rows", []):
        fixed = row.get("fixedCells", [])
        cells = row.get("cells", [])

        # fixed cells: [rank, team cell]
        rank = fixed[0].get("content") if len(fixed) > 0 else None
        team_cell = fixed[1] if len(fixed) > 1 else {}
        team_name = team_cell.get("content")
        team_id = team_cell.get("teamId")

        ti = team_info.get(team_id, {})
        record = {
            "gw": period,
            "rank": rank,
            "teamId": team_id,
            "team": team_name,
            "shortName": ti.get("shortName"),
            "logo": ti.get("logoUrl512"),
            # header order for cells: W, D, L, Points, Win%, WW, FPtsF, FPtsA, Streak
            "W": cells[0].get("content") if len(cells) > 0 else None,
            "D": cells[1].get("content") if len(cells) > 1 else None,
            "L": cells[2].get("content") if len(cells) > 2 else None,
            "Points": cells[3].get("content") if len(cells) > 3 else None,
            "Win%": cells[4].get("content") if len(cells) > 4 else None,
            "WW": cells[5].get("content") if len(cells) > 5 else None,
            "FPtsF": cells[6].get("content") if len(cells) > 6 else None,
            "FPtsA": cells[7].get("content") if len(cells) > 7 else None,
            "Streak": cells[8].get("content") if len(cells) > 8 else None,
        }
        records.append(record)

df = pd.DataFrame(records)

# make numeric columns numeric
for col in ["rank", "W", "D", "L", "Points", "Win%", "WW", "FPtsF", "FPtsA"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [20]:
df[df.gw == 5]

,gw,rank,teamId,team,shortName,logo,W,D,L,Points,Win%,WW,FPtsF,FPtsA,Streak
40,5,1,nbnr5fcnmdl0krp3,Cheasle FC,CFC,https://fantraximg.com/logos/vx5/tmLogo_vx5na4...,5,0,0,15,1.0,2,673.0,508.5,5 (W)
41,5,2,aoj4p4rlmdl0krp3,MBruno's AmAdventures,MGintjee,https://fantraximg.com/logos/l0x/tmLogo_l0x5w1...,4,0,1,12,0.8,10,724.5,626.5,4 (W)
42,5,3,4ww02m1rmdl0krp3,Morning Timber,MT,https://fantraximg.com/assets/images/icons/fan...,4,0,1,12,0.8,6,715.0,590.0,3 (W)
43,5,4,71z5x96fmdl0krp3,Benford FC,BenMtz,https://fantraximg.com/logos/fl4/tmLogo_fl4rn9...,3,0,2,9,0.6,9,736.0,722.5,1 (L)
44,5,5,q0m6hwmimdl0krp3,GAK-PO-TAY-TOES,Taters,https://fantraximg.com/logos/794/tmLogo_794fqa...,3,0,2,9,0.6,5,720.5,597.5,1 (L)
45,5,6,ipm424ppmdl0krp3,Magpies United,NU,https://fantraximg.com/logos/vxl/tmLogo_vxlsmx...,2,0,3,6,0.4,8,663.5,697.5,2 (L)
46,5,7,frxhyucbmdl0krp3,Seanhampton,SHFC,https://fantraximg.com/logos/d0i/tmLogo_d0iynm...,2,0,3,6,0.4,7,655.0,690.0,1 (W)
47,5,8,xngo7h0mmdl0krp3,Thottenham Hotsluts,THHS,https://fantraximg.com/logos/v95/tmLogo_v95dat...,2,0,3,6,0.4,4,646.0,681.0,1 (W)
48,5,9,bdh1g8b8mdl0krp3,Walton Goggonzola,Zac,https://fantraximg.com/assets/images/icons/fan...,0,0,5,0,0.0,1,539.0,753.0,5 (L)
49,5,10,nax64rt3mdl0krp3,FPL 5: AutoPick Strikes Back,APSB,https://fantraximg.com/assets/images/icons/fan...,0,0,5,0,0.0,3,512.5,718.5,5 (L)


In [25]:
df_trim = df[
    df.gw <= latest_gw
][['W', 'D', 'L','Points','Win%', 'WW', 'FPtsF', 'FPtsA', 'Streak','team','rank','gw']]

In [26]:
data = df_trim

chart = alt.Chart(data).mark_line(point=True).encode(
    x = alt.X('gw', scale=alt.Scale(domain=[0, 38]), title="Gameweek"),
    y=alt.Y('rank', aggregate={'argmax': 'gw'}, scale=alt.Scale(domain=[10,1]), title="Rank"),
    color=alt.Color("team", legend=None),
).transform_window(
    rank="rank()",
    sort=[alt.SortField("rank", order="ascending")],
    groupby=["gw"]
).properties(
    title="Fake Internet Soccer XIV standings by gameweek",
    width=700,
    height=350,
)

labels = alt.Chart(data).mark_text(
    align='left', dx=5
).encode(
    x = alt.X('max(gw)', scale=alt.Scale(domain=[0, 38])),
    y=alt.Y('rank', aggregate={'argmax': 'gw'}, scale=alt.Scale(domain=[10,1])),
    text='team:N',
    color='team:N',
).transform_window(
    rank="rank()",
    sort=[alt.SortField("rank", order="ascending")],
    groupby=["gw"]
)

chart + labels

alt.LayerChart(...)

In [28]:
df_trim.to_csv(f"data/output/standings/standings-season-{season}.csv", index=False)

In [29]:
def pivot2datawrapper(df, f):
    pivot_df = pd.pivot(df, index='gw', columns='team', values='rank').reset_index()
    pivot_df.to_csv(f'data/output/datawrapper/{f}', index=False)

In [31]:
pivot2datawrapper(df_trim, 'season-standings-15.csv')